# Healthy vs sick patients data preparation

## Merge the two datasets

In [1]:
import pandas as pd

In [2]:
# Chemins des fichiers
file1 = "/Users/loictalignani/research/project/lipidomics/data/pretreatment/healthy_vs_sick_patients/Healthy_patients.tsv"
file2 = "/Users/loictalignani/research/project/lipidomics/data/pretreatment/healthy_vs_sick_patients/MISSE_Global_Lipidomics_results_selected_patients_D0.tsv"

In [3]:
# Lecture des fichiers TSV
df1 = pd.read_csv(file1, sep='\t')
df2 = pd.read_csv(file2, sep='\t')

In [4]:
# Vérification du nom de la première colonne (feature)
col_name1 = df1.columns[0]
col_name2 = df2.columns[0]
print(f"\nNom de la colonne feature dans fichier 1: '{col_name1}'")
print(f"Nom de la colonne feature dans fichier 2: '{col_name2}'")


Nom de la colonne feature dans fichier 1: 'feature'
Nom de la colonne feature dans fichier 2: 'feature'


In [5]:
# Affichage des informations sur les dataframes
print(f"Fichier 1: {df1.shape[0]} lignes, {df1.shape[1]} colonnes")
print(f"Fichier 2: {df2.shape[0]} lignes, {df2.shape[1]} colonnes")

Fichier 1: 146 lignes, 21 colonnes
Fichier 2: 163 lignes, 21 colonnes


In [6]:
# Vérification du nom de la première colonne (feature)
col_name1 = df1.columns[0]
col_name2 = df2.columns[0]
print(f"\nNom de la colonne feature dans fichier 1: '{col_name1}'")
print(f"Nom de la colonne feature dans fichier 2: '{col_name2}'")


Nom de la colonne feature dans fichier 1: 'feature'
Nom de la colonne feature dans fichier 2: 'feature'


In [7]:
# Nettoyage des données AVANT la fusion
print("\n" + "="*60)
print("NETTOYAGE DES DONNÉES")
print("="*60)

# 1. Remplacer les virgules par des points-virgules dans la colonne feature
virgules_in_features_df1 = df1[col_name1].str.contains(',', na=False).sum()
virgules_in_features_df2 = df2[col_name2].str.contains(',', na=False).sum()

print(f"\nVirgules détectées dans les noms de features:")
print(f"  - Fichier 1: {virgules_in_features_df1} lignes concernées")
print(f"  - Fichier 2: {virgules_in_features_df2} lignes concernées")


NETTOYAGE DES DONNÉES

Virgules détectées dans les noms de features:
  - Fichier 1: 8 lignes concernées
  - Fichier 2: 0 lignes concernées


In [8]:
if virgules_in_features_df1 > 0:
    df1[col_name1] = df1[col_name1].str.replace(',', ';')
    print(f"  → Virgules remplacées par ';' dans fichier 1")

if virgules_in_features_df2 > 0:
    df2[col_name2] = df2[col_name2].str.replace(',', ';')
    print(f"  → Virgules remplacées par ';' dans fichier 2")

  → Virgules remplacées par ';' dans fichier 1


In [9]:
# 2. Remplacer les virgules par des points dans les colonnes numériques
# Pour toutes les colonnes sauf la colonne feature
numeric_cols_df1 = [col for col in df1.columns if col != col_name1]
numeric_cols_df2 = [col for col in df2.columns if col != col_name2]

print(f"\nConversion des virgules décimales en points:")
print(f"  - Fichier 1: {len(numeric_cols_df1)} colonnes numériques à traiter")
print(f"  - Fichier 2: {len(numeric_cols_df2)} colonnes numériques à traiter")

# Fonction pour convertir les valeurs avec virgule en float


def convert_comma_to_float(value):
    if pd.isna(value):
        return value
    if isinstance(value, str):
        return float(value.replace(',', '.'))
    return value


# Application de la conversion pour df1
for col in numeric_cols_df1:
    df1[col] = df1[col].apply(convert_comma_to_float)

# Application de la conversion pour df2
for col in numeric_cols_df2:
    df2[col] = df2[col].apply(convert_comma_to_float)

print("  → Conversion terminée")

print("\nAperçu après nettoyage:")
print("Fichier 1:")
print(df1.head(3))
print("\nFichier 2:")
print(df2.head(3))


Conversion des virgules décimales en points:
  - Fichier 1: 20 colonnes numériques à traiter
  - Fichier 2: 20 colonnes numériques à traiter
  → Conversion terminée

Aperçu après nettoyage:
Fichier 1:
   feature  JV-015-RAN-IC-M2  BS-007-RAN-01-M3  BS-007-RAN-02-M3  \
0  DG 30:0              0.36              0.36              0.00   
1  DG 32:0              1.84              1.88              1.55   
2  DG 32:1              1.53              2.09              1.77   

   KT-004-SOC-16-M1  BS-135-RAN-IC-M3  JV-070-DAV-07-M3  BS-129-RAT-10-M5  \
0              0.00              0.51              0.40              0.00   
1              0.82              2.16              1.91              1.65   
2              1.53              3.01              1.21              1.01   

   JV-036-SOC-10-M2  BS-126-RAT-05-M2  ...  JV-070-DAV-04-M1  \
0              0.64              0.00  ...              0.00   
1              2.05              1.28  ...              0.51   
2              2.21     

In [10]:
# Fusion des deux dataframes sur la colonne feature (inner join)
# Cela ne garde que les features présentes dans les deux fichiers
df_merged = pd.merge(df1, df2, on=col_name1, how='inner', suffixes=('_healthy', '_D0'))

In [11]:
print(
    f"\nAprès fusion: {df_merged.shape[0]} lignes, {df_merged.shape[1]} colonnes")
print(f"Nombre de features communes: {df_merged.shape[0]}")


Après fusion: 145 lignes, 41 colonnes
Nombre de features communes: 145


In [12]:
# Sauvegarde du résultat
output_file = "/Users/loictalignani/research/project/lipidomics/data/pretreatment/healthy_vs_sick_patients/healthy_sick_lipidomics.tsv"
df_merged.to_csv(output_file, sep='\t', index=False)

print(f"\nFichier fusionné sauvegardé: {output_file}")


Fichier fusionné sauvegardé: /Users/loictalignani/research/project/lipidomics/data/pretreatment/healthy_vs_sick_patients/healthy_sick_lipidomics.tsv


In [13]:
# Bilan des lignes non fusionnées
print("\n" + "="*60)
print("BILAN DES LIGNES NON FUSIONNÉES")
print("="*60)

# Features présentes uniquement dans df1
features_df1 = set(df1[col_name1])
features_df2 = set(df2[col_name2])
features_merged = set(df_merged[col_name1])

only_in_df1 = features_df1 - features_df2
only_in_df2 = features_df2 - features_df1

print(f"\nFeatures uniquement dans Healthy_patients.tsv: {len(only_in_df1)}")
if len(only_in_df1) > 0:
    print("Exemples:")
    for feature in list(only_in_df1):
        print(f"  - {feature}")


BILAN DES LIGNES NON FUSIONNÉES

Features uniquement dans Healthy_patients.tsv: 1
Exemples:
  - DG 30:0


In [14]:
print(
    f"\nFeatures uniquement dans MISSE_Global_Lipidomics_results_selected_patients_D0.tsv: {len(only_in_df2)}")
if len(only_in_df2) > 0:
    print("Exemples:")
    for feature in list(only_in_df2):
        print(f"  - {feature}")


print(f"\nRésumé:")
print(f"  - Total features dans df1: {len(features_df1)}")
print(f"  - Total features dans df2: {len(features_df2)}")
print(f"  - Features communes (fusionnées): {len(features_merged)}")
print(f"  - Features perdues (total): {len(only_in_df1) + len(only_in_df2)}")
print(
    f"  - Taux de fusion: {len(features_merged)/max(len(features_df1), len(features_df2))*100:.1f}%")


Features uniquement dans MISSE_Global_Lipidomics_results_selected_patients_D0.tsv: 19
Exemples:
  - TG 58:5
  - FA 22:6
  - TG 52:4
  - SM 40:0;O2
  - TG 52:1
  - SM 34:0;O2
  - TG 52:5
  - PC 34:0;O2
  - LPC 13:0
  - TG 49:2
  - PC 36:3;P/36:3;O
  - PE 38:4
  - TG 58:11
  - TG 52:2
  - TG 51:1
  - CAR 3:0
  - CAR 8:0
  - CAR 2:0
  - SM 36:0;O2

Résumé:
  - Total features dans df1: 145
  - Total features dans df2: 163
  - Features communes (fusionnées): 144
  - Features perdues (total): 20
  - Taux de fusion: 88.3%


## Group information table file preparation

In [15]:
# Liste des patients sick avec leur groupe (mild ou severe)
sick_patients = {
    'BS-082D0': 'mild',
    'BS-364D0': 'mild',
    'KT-193D0': 'severe',
    'KT-247D0': 'severe',
    'KT-312D0': 'severe',
    'KT-313D0': 'mild',
    'KT-347D0': 'severe',
    'KT-412D0': 'mild',
    'KT-417D0': 'severe',
    'KT-445D0': 'mild',
    'KT-522D0': 'severe',
    'KT-525D0': 'severe',
    'KT-537D0': 'mild',
    'KT-538D0': 'mild',
    'KT-539D0': 'mild',
    'KT-565D0': 'severe',
    'KT-663D0': 'mild',
    'KT-695D0': 'mild',
    'KT-705D0': 'severe',
    'KT-723D0': 'mild'
}

In [16]:
# Récupération des noms de colonnes (patients) de df1, en excluant la colonne feature
healthy_patients = [col for col in df1.columns if col != col_name1]

In [17]:
# Création de la liste pour le dataframe
group_data = []

# Ajout des patients sick
for patient, severity in sick_patients.items():
    group_data.append({
        'sample_name': patient,
        'label_name': patient,
        'group': severity,
        'pair': 'NA'
    })

# Ajout des patients healthy
for patient in healthy_patients:
    group_data.append({
        'sample_name': patient,
        'label_name': patient,
        'group': 'healthy',
        'pair': 'NA'
    })

In [18]:
# Création du dataframe
df_groups = pd.DataFrame(group_data)

# Sauvegarde du fichier
group_output_file = "/Users/loictalignani/research/project/lipidomics/data/pretreatment/healthy_vs_sick_patients/group_information_table_healthy_vs_sick_patients_D0.tsv"
df_groups.to_csv(group_output_file, sep='\t', index=False)

print(f"\nFichier de groupes créé: {group_output_file}")
print(f"Nombre total d'échantillons: {len(df_groups)}")
print(f"  - Patients healthy: {len(healthy_patients)}")
print(
    f"  - Patients mild: {sum(1 for v in sick_patients.values() if v == 'mild')}")
print(
    f"  - Patients severe: {sum(1 for v in sick_patients.values() if v == 'severe')}")

print("\nAperçu du fichier de groupes:")
print(df_groups.head(10))


Fichier de groupes créé: /Users/loictalignani/research/project/lipidomics/data/pretreatment/healthy_vs_sick_patients/group_information_table_healthy_vs_sick_patients_D0.tsv
Nombre total d'échantillons: 40
  - Patients healthy: 20
  - Patients mild: 11
  - Patients severe: 9

Aperçu du fichier de groupes:
  sample_name label_name   group pair
0    BS-082D0   BS-082D0    mild   NA
1    BS-364D0   BS-364D0    mild   NA
2    KT-193D0   KT-193D0  severe   NA
3    KT-247D0   KT-247D0  severe   NA
4    KT-312D0   KT-312D0  severe   NA
5    KT-313D0   KT-313D0    mild   NA
6    KT-347D0   KT-347D0  severe   NA
7    KT-412D0   KT-412D0    mild   NA
8    KT-417D0   KT-417D0  severe   NA
9    KT-445D0   KT-445D0    mild   NA
